# Étape 2 : Optimisation de l'Agent RL

## Objectif
Améliorer l'agent d'apprentissage par renforcement pour qu'il dépasse de manière stable une récompense moyenne de **200 points** sur 100 épisodes d'évaluation.

## Prérequis
- ✓ Modèle de référence établi à l'étape 1
- ✓ Dataset et environnement prêts
- ✓ Infrastructure d'entraînement en place

## Méthodologie
- Modifier **un seul hyperparamètre à la fois** pour isoler son effet
- Documenter chaque expérience de façon systématique
- Utiliser TensorBoard pour visualiser les courbes de récompense
- Sauvegarder les modèles prometteurs
- S'assurer que le score > 200 est **stable** (faible écart-type)

In [5]:
# Imports et configuration
import os, datetime
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import tensorflow as tf


# Load the TensorBoard notebook extension
%load_ext tensorboard

# Ajouter le chemin src au path si nécessaire
project_root = Path().cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# Créer les répertoires pour les logs et modèles
log_dir = project_root / "data" / "logs"
model_dir = project_root / "models"
log_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

print(f"Répertoire des logs: {log_dir}")
print(f"Répertoire des modèles: {model_dir}")

Répertoire des logs: /Users/xaviercoulon/Documents/OC/OC_P11_Entrainement_Agent_RL/data/logs
Répertoire des modèles: /Users/xaviercoulon/Documents/OC/OC_P11_Entrainement_Agent_RL/models


## Tableau de Suivi des Expériences

Ce tableau synthétise toutes les expériences menées et facilite la comparaison :

| N° Exp | Agent | Paramètre modifié | Valeur | Score moyen | Écart-type | Stabilité | Statut | Notes |
|--------|-------|-------------------|--------|-------------|-----------|-----------|--------|-------|
| 0 (ref) | PPO/DQN | - | - | ? | ? | ? | baseline | À compléter avec résultats étape 1 |
| 1 | DQN | learning_rate | 0.0005 | ? | ? | ? | en cours | Conservative LR pour stabilité |
| 2 | DQN | learning_rate | 0.001 | ? | ? | ? | pending | À tester si exp_1 < 200 |

*Stabilité : Considérée comme stable si écart-type < 10-15 points*

## Utilisation de TensorBoard

TensorBoard permet de visualiser et comparer les courbes de récompense de tes différentes expériences.

### Lancer TensorBoard
```bash
tensorboard --logdir=../data/logs --port=6006
```
Puis accéder à http://localhost:6006

### Organisation des logs
- Chaque expérience doit avoir son propre dossier de logs
- Exemple : `logs/exp_1_learning_rate_0.001/`, `logs/exp_2_gamma_0.95/`
- TensorBoard affichera automatiquement toutes les courbes sous forme comparative

### Avantages
✓ Visualisation en temps réel de l'entraînement
✓ Comparaison facile entre plusieurs runs
✓ Analyse des métriques (récompense, loss, etc.)
✓ Export des graphiques pour la documentation

## Utilities pour le suivi des expériences

In [6]:
# Fonction pour créer une expérience structurée
class ExperimentTracker:
    """Classe pour documenter et tracker les expériences"""
    
    def __init__(self, exp_name, agent_type, modified_param, param_value, baseline_score=None):
        """
        Args:
            exp_name: nom de l'expérience (ex: "exp_1_learning_rate")
            agent_type: PPO ou DQN
            modified_param: paramètre modifié (ex: "learning_rate")
            param_value: valeur du paramètre
            baseline_score: score de référence pour comparaison
        """
        self.exp_name = exp_name
        self.agent_type = agent_type
        self.modified_param = modified_param
        self.param_value = param_value
        self.baseline_score = baseline_score
        self.exp_dir = log_dir / exp_name
        self.exp_dir.mkdir(parents=True, exist_ok=True)
        self.results = {}
        
    def log_result(self, score_mean, score_std, num_episodes=100):
        """Enregistrer les résultats de l'expérience"""
        self.results = {
            'exp_name': self.exp_name,
            'agent_type': self.agent_type,
            'modified_param': self.modified_param,
            'param_value': str(self.param_value),
            'score_mean': score_mean,
            'score_std': score_std,
            'num_episodes': num_episodes,
            'timestamp': datetime.now().isoformat(),
            'succeeded': score_mean > 200 and score_std < 15
        }
        return self.results
    
    def print_summary(self):
        """Afficher un résumé de l'expérience"""
        if not self.results:
            print("Pas de résultats enregistrés encore")
            return
        
        print(f"\n{'='*60}")
        print(f"Expérience: {self.exp_name}")
        print(f"Agent: {self.agent_type} | Paramètre: {self.modified_param} = {self.param_value}")
        print(f"{'='*60}")
        print(f"Score moyen: {self.results['score_mean']:.2f} ± {self.results['score_std']:.2f}")
        print(f"Objectif: > 200 (écart-type < 15)")
        print(f"✓ RÉUSSI" if self.results['succeeded'] else "✗ Non atteint")
        if self.baseline_score:
            diff = self.results['score_mean'] - self.baseline_score
            print(f"Amélioration vs baseline: {diff:+.2f} ({(diff/self.baseline_score)*100:+.1f}%)")
        print(f"{'='*60}\n")

# Créer un DataFrame global pour tracker toutes les expériences
experiments_df = pd.DataFrame(columns=[
    'exp_name', 'agent_type', 'modified_param', 'param_value', 
    'score_mean', 'score_std', 'num_episodes', 'timestamp', 'succeeded'
])

print("✓ Utilities de suivi chargées")

✓ Utilities de suivi chargées


---

# Expériences

## Template pour chaque expérience

Pour chaque nouvel essai, suivre ce template :

### Description
- Paramètre modifié : [ex: learning_rate]
- Valeur : [ex: 0.001]
- Hypothèse : [pourquoi ce changement devrait améliorer la performance]
- Modifications du code : [détails des changements apportés]

### Code d'entraînement
- Configuration de l'agent avec le nouveau paramètre
- Entraînement et logging
- Sauvegarde du modèle

### Résultats et analyse
- Score moyen et écart-type
- Courbe TensorBoard (capture d'écran ou lien)
- Observations : stabilité, convergence, surprises
- Comparaison avec la baseline

### Conclusion
- Paramètre accepté/rejeté pour l'itération suivante
- Prochaine expérience recommandée

## Expérience 1 : Optimisation du Learning Rate pour DQN

### Description
- **Paramètre modifié** : `learning_rate`
- **Valeur testée** : `0.0005` (au lieu de la valeur par défaut)
- **Agent** : DQN (Deep Q-Network)
- **Hypothèse** : Un learning rate plus faible permet généralement une meilleure convergence et une plus grande stabilité lors de l'apprentissage sur des problèmes de contrôle. Nous testons ici une valeur conservative pour stabiliser les gradients.
- **Dossier logs** : `exp_1_dqn_lr_0.0005`
- **Baseline référence** : À récupérer depuis l'étape 1

In [7]:
# Expérience 1 : DQN avec Learning Rate réduit (stable-baselines3)
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

# 1. Initialiser le tracker
exp_1 = ExperimentTracker(
    exp_name="exp_1_dqn_lr_0.0005",
    agent_type="DQN",
    modified_param="learning_rate",
    param_value=0.0005,
    baseline_score=None  # À remplir avec le score de référence de l'étape 1
)

print(f"🚀 Lancement de l'Expérience 1: DQN avec lr={exp_1.param_value}")
print(f"📁 Logs seront sauvegardés dans: {exp_1.exp_dir}\n")

# 2. Créer l'environnement
try:
    env = gym.make("LunarLander-v3")
    print("✓ Environnement LunarLander-v3 créé\n")
    
    # 3. Créer et entraîner le modèle DQN avec learning_rate réduit
    print("⏳ Création et entraînement du modèle DQN...")
    num_training_steps = 50000  # À adapter selon ton besoin
    
    model = DQN(
        "MlpPolicy",
        env,
        learning_rate=0.0005,        # ← PARAMÈTRE TESTÉ
        gamma=0.99,                  # discount factor
        exploration_fraction=0.1,    # durée de la phase exploration
        exploration_initial_eps=1.0,
        exploration_final_eps=0.01,
        batch_size=64,
        buffer_size=10000,
        target_update_interval=1000,
        verbose=0,
        tensorboard_log=str(exp_1.exp_dir)
    )
    
    # Entraîner le modèle
    model.learn(total_timesteps=num_training_steps)
    print(f"✓ Entraînement terminé ({num_training_steps} timesteps)\n")
    
    # 4. Évaluer sur 100 épisodes
    print("📊 Évaluation sur 100 épisodes...")
    eval_rewards, eval_std = evaluate_policy(
        model, 
        env, 
        n_eval_episodes=100,
        deterministic=True  # Sans exploration aléatoire
    )
    
    score_mean = eval_rewards
    score_std = eval_std
    print(f"Résultats: {score_mean:.2f} ± {score_std:.2f}\n")
    
    # 5. Enregistrer les résultats
    exp_1.log_result(score_mean, score_std, num_episodes=100)
    exp_1.print_summary()
    
    # 6. Sauvegarder le modèle si prometteur
    if exp_1.results['succeeded']:
        model_path = model_dir / f"exp_1_dqn_lr_0.0005_successful_{score_mean:.0f}"
        model.save(str(model_path))
        print(f"✓ Modèle RÉUSSI sauvegardé: {model_path}")
    else:
        print(f"⚠️  Score non atteint")
        print(f"   Objectif: > 200, Obtenu: {score_mean:.2f}")
        print(f"   Écart-type: {score_std:.2f} (objectif: < 15)")
    
    # 7. Ajouter au DataFrame global
    experiments_df_new = pd.DataFrame([exp_1.results])
    globals()['experiments_df'] = pd.concat([experiments_df, experiments_df_new], ignore_index=True)
    print(f"\n✓ Résultats ajoutés au tableau de suivi")
    
    # Fermer l'environnement
    env.close()

except Exception as e:
    print(f"❌ Erreur lors de l'expérience 1:")
    print(f"   {type(e).__name__}: {e}")
    print(f"\n💡 Vérifier:")
    print(f"   - stable_baselines3 est installé: pip install stable-baselines3")
    print(f"   - gymnasium est installé: pip install gymnasium")
    print(f"   - L'environnement LunarLander-v3 est disponible")

🚀 Lancement de l'Expérience 1: DQN avec lr=0.0005
📁 Logs seront sauvegardés dans: /Users/xaviercoulon/Documents/OC/OC_P11_Entrainement_Agent_RL/data/logs/exp_1_dqn_lr_0.0005

✓ Environnement LunarLander-v3 créé

⏳ Création et entraînement du modèle DQN...
✓ Entraînement terminé (50000 timesteps)

📊 Évaluation sur 100 épisodes...


/Users/xaviercoulon/Documents/OC/OC_P11_Entrainement_Agent_RL/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Résultats: -55.64 ± 31.39


Expérience: exp_1_dqn_lr_0.0005
Agent: DQN | Paramètre: learning_rate = 0.0005
Score moyen: -55.64 ± 31.39
Objectif: > 200 (écart-type < 15)
✗ Non atteint

⚠️  Score non atteint
   Objectif: > 200, Obtenu: -55.64
   Écart-type: 31.39 (objectif: < 15)

✓ Résultats ajoutés au tableau de suivi


### Résultats Expérience 1

**Après exécution de l'expérience, compléter :**
- Score moyen : ? ± ?
- Stabilité : ✓ Accepté / ✗ Rejeté
- Convergence : [Converge rapidement / Lentement / Instable]
- Observations : 
  - [Notes sur le comportement d'apprentissage]
  - [Évolution de la récompense au fil du temps]
  - [Époque à laquelle la stabilité est atteinte]
- Action recommandée : [Garder ce LR / Augmenter / Diminuer / Tester autre paramètre]
- Captures TensorBoard : [Créer des screenshots des courbes et les insérer]

---

# Comparaison des Expériences

## Tableau récapitulatif final

In [ ]:
# Afficher le tableau de toutes les expériences
print("Résumé de toutes les expériences :")
print(experiments_df.to_string(index=False))

# Identifier le meilleur modèle
if len(experiments_df) > 0:
    best_idx = experiments_df['score_mean'].idxmax()
    best_exp = experiments_df.loc[best_idx]
    print(f"\n✓ Meilleur modèle : {best_exp['exp_name']} avec un score de {best_exp['score_mean']:.2f}")

## Comparaison visuelle

In [ ]:
# Visualiser la comparaison des expériences
def plot_experiments_comparison():
    """Créer des graphiques de comparaison des expériences"""
    if len(experiments_df) == 0:
        print("Aucune expérience à comparer")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Graphique 1 : Score moyen par expérience
    ax = axes[0]
    exp_names = experiments_df['exp_name']
    scores = experiments_df['score_mean']
    stds = experiments_df['score_std']
    
    ax.bar(range(len(exp_names)), scores, yerr=stds, capsize=5, alpha=0.7, color='steelblue')
    ax.axhline(y=200, color='green', linestyle='--', label='Objectif (200)', linewidth=2)
    ax.set_xlabel('Expérience')
    ax.set_ylabel('Score moyen')
    ax.set_title('Comparaison des scores moyens par expérience')
    ax.set_xticks(range(len(exp_names)))
    ax.set_xticklabels(exp_names, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Graphique 2 : Impact du paramètre sur le score
    ax = axes[1]
    ax.scatter(range(len(exp_names)), scores, s=100, alpha=0.6, color='coral')
    ax.plot(range(len(exp_names)), scores, 'o-', alpha=0.3)
    ax.axhline(y=200, color='green', linestyle='--', label='Objectif (200)', linewidth=2)
    ax.set_xlabel('Expérience')
    ax.set_ylabel('Score moyen')
    ax.set_title('Évolution du score across experiments')
    ax.set_xticks(range(len(exp_names)))
    ax.set_xticklabels(exp_names, rotation=45, ha='right')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Exemple : plot_experiments_comparison()
print("✓ Fonction de visualisation chargée")

---

# Analyse et Conclusions

## Hyperparamètres efficaces

[À remplir après les expériences]
- Paramètre 1 : Valeur optimale identifiée ?
- Paramètre 2 : Valeur optimale identifiée ?

## Score final atteint

| Métrique | Valeur |
|----------|--------|
| Score moyen | ? |
| Écart-type | ? |
| Objectif (200) | ✓ Atteint / ✗ Non atteint |

## Observations clés

- [Paramètre X a eu l'impact le plus significatif]
- [Exploration Y a révélé ...]
- [Surprise ou apprentissage intéressant]

## Pistes futures

1. [Paramètre suivant à explorer]
2. [Architecture alternative à tester]
3. [Ajustement fin recommandé]

## Modèle final recommandé

**Meilleur modèle** : [nom du modèle]  
**Localisation** : [chemin vers le fichier sauvegardé]  
**Score** : ? ± ?